<a href="https://colab.research.google.com/github/alfredqbit/type-named-tensor-ml/blob/main/type_named_tensor_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Typed Named Tensors — companion notebook

> **Companion to:** *A Typed Named Tensor Notation for Deep Learning.*

This notebook is the executable counterpart to the paper. It walks through
the type discipline of §2–§3 cell by cell, builds the framework as Python,
and ends with the §3.4 worked example: the type checker rejecting the
documented BatchNorm-information-leak bug from MoCo and accepting each of
the three canonical fixes.

**What you'll see (and run):**

1. The sort algebra (§2) — twelve sorts encoding what each tensor axis is *for*
2. Refinement subsorts (§2.5) — narrowing legality via a partial order
3. The legality table (§2.3) — eight elementary operations × twelve sorts, derived from the axioms
4. Typed named tensors (§3) — sorted axes plus a NumPy backend
5. Primitive operations — `contract`, `reduce`, `normalize`, `concat`, `attention`, each with type checking
6. Basic type-checking sanity tests
7. The §3.4 MoCo worked example as a runnable narrative
8. A small Transformer-block sketch (§6.1) as an integration test

**Reading mode.** Each section has prose first, then code. Run cells in order.
The whole notebook executes in well under a second on any modern laptop —
the type-checking step itself is microseconds; everything else is light NumPy.

**Honest limits.** This is a proof-of-concept reference implementation
(~530 lines), not a competitor to PyTorch. There is no autodiff (the
backward pass is proved on paper, not run); no GPU; the operation set
covers what's needed to demonstrate the type discipline, not what's
needed to train a real model. The point is decidability and concrete
bug-catching.

In [1]:
import torch
import math
from dataclasses import dataclass
from typing import Optional, Iterable
from enum import Enum

# --- 1. The sort algebra ---
class Sort(Enum):
    B = ("batch",   "S_n")
    T = ("token",   "trivial")
    F = ("feature", "matched")
    H = ("head",    "S_n")
    K = ("key",     "matched")
    V = ("vocab",   "matched")
    C = ("channel", "matched")
    S = ("spatial", "trivial")
    L = ("layer",   "trivial")
    G = ("group",   "S_n")
    N = ("noise",   "S_n")
    M = ("memory",  "trivial")

    def __init__(self, name: str, group: str):
        self._name_ = name
        self.group = group

# --- 2. Refinements ---
@dataclass(frozen=True)
class Refinement:
    parent: Sort
    tag: str

    def __le__(self, other) -> bool:
        if isinstance(other, Sort):
            return self.parent == other
        return self.parent == other.parent and self.tag == other.tag

    def __str__(self) -> str:
        return f"{self.parent.name}^{self.tag}"

B_iid    = Refinement(Sort.B, "iid")
B_paired = Refinement(Sort.B, "paired")
B_seq    = Refinement(Sort.B, "seq")
T_causal = Refinement(Sort.T, "causal")
T_set    = Refinement(Sort.T, "set")
N_iid    = Refinement(Sort.N, "iid")
N_pi     = Refinement(Sort.N, "pi")

def basic(s) -> Sort:
    return s.parent if isinstance(s, Refinement) else s

# --- 3. Sorted axes ---
@dataclass(frozen=True)
class Axis:
    name: str
    sort: object
    dim: int

    def __str__(self) -> str:
        return f"{self.name}:{self.sort}={self.dim}"

# --- 4. Legality table ---
def can_reduce(s, op: str, licensed_in: str = "") -> bool:
    bs = basic(s)
    if bs == Sort.B: return op == "loss" or licensed_in == "loss"
    if bs == Sort.T: return op in {"mean", "max"} and licensed_in == "pooling"
    if bs == Sort.F: return True
    if bs == Sort.H: return op == "mean"
    if bs == Sort.K: return op == "sum"
    if bs == Sort.V: return op in {"logsumexp", "argmax", "softmax"}
    if bs == Sort.C: return True
    if bs == Sort.S: return op in {"mean", "max", "sum"} and licensed_in in {"pooling", "conv"}
    if bs == Sort.L: return licensed_in == "deep_ensemble"
    if bs == Sort.G: return op in {"sum", "mean"} and licensed_in in {"gating", "moe"}
    if bs == Sort.N: return op == "expectation" and licensed_in in {"loss", "marginalize"}
    if bs == Sort.M: return licensed_in == "state_readout"
    return False

def can_contract(s, licensed_in: str = "") -> bool:
    bs = basic(s)
    if bs in {Sort.B, Sort.V, Sort.N, Sort.L}: return False
    if bs == Sort.F or bs == Sort.C: return True
    if bs == Sort.K: return licensed_in == "attention"
    if bs == Sort.T: return licensed_in in {"attention", "conv"}
    if bs == Sort.S: return licensed_in == "conv"
    if bs == Sort.H: return licensed_in == "head_mix"
    if bs == Sort.G: return licensed_in == "gated_mix"
    if bs == Sort.M: return licensed_in == "recurrence"
    return False

def can_concat(s) -> bool:
    bs = basic(s)
    return bs in {Sort.F, Sort.C, Sort.H, Sort.V, Sort.T, Sort.S, Sort.L, Sort.G}

ALWAYS_ALIGN = {Sort.B, Sort.H, Sort.G, Sort.N, Sort.L}

# --- 5. Type errors ---
class TypeError_(Exception):
    def __init__(self, message: str, axis: Optional[Axis] = None):
        super().__init__(message)
        self.axis = axis

In [2]:
# --- 6. PyTorch Typed Named Tensors ---
@dataclass
class TorchTNT:
    data: torch.Tensor
    axes: tuple[Axis, ...]

    def __post_init__(self):
        if self.data.shape != tuple(a.dim for a in self.axes):
            raise ValueError(f"shape {self.data.shape} doesn't match axes {self.axes}")
        if len(set(a.name for a in self.axes)) != len(self.axes):
            raise ValueError(f"duplicate axis names in {self.axes}")

    def __repr__(self):
        sig = ", ".join(str(a) for a in self.axes)
        req_grad = ", requires_grad=True" if self.data.requires_grad else ""
        dev = f", device='{self.device}'" if self.device.type != 'cpu' else ""
        return f"TorchTNT[{sig}{req_grad}{dev}]"

    @property
    def device(self) -> torch.device:
        return self.data.device

    def to(self, device) -> "TorchTNT":
        return TorchTNT(self.data.to(device), self.axes)

    def signature(self) -> dict[str, Axis]:
        return {a.name: a for a in self.axes}

    def axis(self, name: str) -> Axis:
        return self.signature()[name]

    def _index(self, name: str) -> int:
        for i, a in enumerate(self.axes):
            if a.name == name:
                return i
        raise KeyError(f"axis {name} not in {self.axes}")

    def transposed_to(self, names: Iterable[str]) -> "TorchTNT":
        names = tuple(names)
        idx = [self._index(n) for n in names]
        new_data = self.data.permute(*idx)
        new_axes = tuple(self.axis(n) for n in names)
        return TorchTNT(new_data, new_axes)

def pt_tensor(data, *axes: Axis, device=None) -> TorchTNT:
    return TorchTNT(torch.as_tensor(data, device=device), tuple(axes))

def pt_zeros(*axes: Axis, requires_grad=False, device=None) -> TorchTNT:
    return TorchTNT(torch.zeros(tuple(a.dim for a in axes), requires_grad=requires_grad, device=device), tuple(axes))

def pt_randn(*axes: Axis, generator=None, requires_grad=False, device=None) -> TorchTNT:
    return TorchTNT(torch.randn(tuple(a.dim for a in axes), generator=generator, requires_grad=requires_grad, device=device), tuple(axes))

In [3]:
# --- 7. PyTorch Operations ---
def pt_contract(X: TorchTNT, Y: TorchTNT, *, licensed_in: str = "") -> TorchTNT:
    sigX, sigY = X.signature(), Y.signature()
    shared = set(sigX) & set(sigY)
    only_X = set(sigX) - shared
    only_Y = set(sigY) - shared

    to_contract, to_align = [], []
    for name in shared:
        ax_x, ax_y = sigX[name], sigY[name]
        if ax_x.dim != ax_y.dim:
            raise TypeError_(f"shape mismatch on shared axis {name}: {ax_x.dim} vs {ax_y.dim}", axis=ax_x)
        if basic(ax_x.sort) != basic(ax_y.sort):
            raise TypeError_(f"sort mismatch on shared axis {name}: {ax_x.sort} vs {ax_y.sort}", axis=ax_x)

        bs = basic(ax_x.sort)
        if bs in ALWAYS_ALIGN:
            to_align.append(name)
        elif can_contract(ax_x.sort, licensed_in=licensed_in):
            to_contract.append(name)
        else:
            raise TypeError_(f"axis {name} cannot be contracted in context '{licensed_in}'", axis=ax_x)

    name_to_letter = {}
    next_letter = [0]
    def letter_for(name):
        if name not in name_to_letter:
            name_to_letter[name] = chr(ord('a') + next_letter[0])
            next_letter[0] += 1
        return name_to_letter[name]

    sub_x = "".join(letter_for(a.name) for a in X.axes)
    sub_y = "".join(letter_for(a.name) for a in Y.axes)
    out_names = list(to_align) + list(only_X) + list(only_Y)
    sub_out = "".join(letter_for(n) for n in out_names)
    eq = f"{sub_x},{sub_y}->{sub_out}"

    out_data = torch.einsum(eq, X.data, Y.data)
    out_axes = tuple([sigX[n] for n in to_align] + [sigX[n] for n in only_X] + [sigY[n] for n in only_Y])
    return TorchTNT(out_data, out_axes)

def pt_reduce(X: TorchTNT, axis_name: str, op: str, *, licensed_in: str = "") -> TorchTNT:
    ax = X.axis(axis_name)
    if not can_reduce(ax.sort, op, licensed_in=licensed_in):
        raise TypeError_(f"reduction '{op}' on axis {axis_name}:{ax.sort} is illegal", axis=ax)

    pos = X._index(axis_name)
    if op == "sum":
        new_data = X.data.sum(dim=pos)
    elif op == "mean":
        new_data = X.data.mean(dim=pos)
    elif op == "max":
        new_data = X.data.max(dim=pos).values
    elif op == "logsumexp":
        new_data = torch.logsumexp(X.data, dim=pos)
    elif op == "softmax":
        new_data = torch.nn.functional.softmax(X.data, dim=pos)
        return TorchTNT(new_data, X.axes)
    elif op in ("loss", "expectation"):
        new_data = X.data.mean(dim=pos)
    elif op == "argmax":
        new_data = X.data.argmax(dim=pos)
    else:
        raise ValueError(f"unknown reduction op: {op}")

    new_axes = tuple(a for a in X.axes if a.name != axis_name)
    return TorchTNT(new_data, new_axes)

def pt_normalize(X: TorchTNT, axis_name: str) -> TorchTNT:
    ax = X.axis(axis_name)
    bs = basic(ax.sort)
    if bs == Sort.B:
        if not (isinstance(ax.sort, Refinement) and ax.sort.tag == "iid"):
            raise TypeError_(f"MoCo BatchNorm leak error: applying BN to paired samples.", axis=ax)
    elif bs not in {Sort.F, Sort.C, Sort.S}:
        raise TypeError_(f"Norm not defined on sort {ax.sort}", axis=ax)

    pos = X._index(axis_name)
    mu = X.data.mean(dim=pos, keepdim=True)
    sigma = X.data.std(dim=pos, keepdim=True, unbiased=False) + 1e-6
    return TorchTNT((X.data - mu) / sigma, X.axes)

def pt_concat(X: TorchTNT, Y: TorchTNT, axis_name: str) -> TorchTNT:
    ax_x, ax_y = X.axis(axis_name), Y.axis(axis_name)
    if not can_concat(ax_x.sort):
        raise TypeError_(f"concat on axis {axis_name} is illegal", axis=ax_x)
    if basic(ax_x.sort) != basic(ax_y.sort):
        raise TypeError_(f"sort mismatch concatenating {axis_name}", axis=ax_x)

    for n, a in X.signature().items():
        if n != axis_name:
            if n not in Y.signature() or Y.axis(n).dim != a.dim:
                raise TypeError_(f"non-concat axis {n} disagrees between operands")

    Y2 = Y.transposed_to([a.name for a in X.axes])
    pos = X._index(axis_name)
    new_data = torch.cat([X.data, Y2.data], dim=pos)
    new_axes = tuple(
        Axis(a.name, a.sort, a.dim + ax_y.dim) if a.name == axis_name else a
        for a in X.axes
    )
    return TorchTNT(new_data, new_axes)

def pt_attention(Q: TorchTNT, K: TorchTNT, V: TorchTNT, *, head: str = "h", t_query: str = "t", t_key: str = "t", key_dim: str = "k", value_dim: str = "v") -> TorchTNT:
    k_ax = Q.axis(key_dim)
    if basic(k_ax.sort) != Sort.K:
        raise TypeError_(f"attention requires key_dim of sort K; got {k_ax.sort}")

    scores = pt_contract(Q, K, licensed_in="attention")
    scale = 1.0 / math.sqrt(k_ax.dim)
    scores = TorchTNT(scores.data * scale, scores.axes)
    weights = pt_reduce(scores, t_key, "softmax", licensed_in="attention")
    out = pt_contract(weights, V, licensed_in="attention")
    return out

In [4]:
# --- 8. Type-checking sanity tests (PyTorch) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running tests on device: {device}")

b = Axis("b", B_iid, 4)
t = Axis("t", Sort.T, 8)
d = Axis("d", Sort.F, 16)
dp = Axis("d_prime", Sort.F, 32)

# Test 1: Linear layer
W, X = pt_randn(d, dp, device=device), pt_randn(b, t, d, device=device)
Y = pt_contract(W, X)
assert {a.name for a in Y.axes} == {"d_prime", "b", "t"}
print("✓ Test 1: linear layer W @ X type-checks.")

# Test 2: Sum_b without loss context is rejected
X = pt_randn(b, d, device=device)
try:
    pt_reduce(X, "b", "sum")
    assert False
except TypeError_:
    pass
print("✓ Test 2: Sum_b without loss context is rejected.")

# Test 3: token contraction requires attention context
X, Y_tensor = pt_randn(b, t, d, device=device), pt_randn(b, t, d, device=device)
try:
    pt_contract(X, Y_tensor)
    assert False
except TypeError_:
    pass
print("✓ Test 3: token-axis contraction rejected outside attention.")

Z = pt_contract(X, Y_tensor, licensed_in="attention")
print("✓ Test 3': token-axis contraction succeeds inside attention.")

Running tests on device: cuda
✓ Test 1: linear layer W @ X type-checks.
✓ Test 2: Sum_b without loss context is rejected.
✓ Test 3: token-axis contraction rejected outside attention.
✓ Test 3': token-axis contraction succeeds inside attention.
